# Exercises 21: exploration of various datasets
In this section we will explore the iris and the forest cover datasets.

For these exercises we will need `pandas`, `numpy`, `matplotlib` and `scikit-learn`. We also import `seaborn` to showcase its usage for certain figures.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pprint import pprint as pp

In [ ]:
pd.set_option('display.max_columns', 1000)
plt.style.use("bmh")

## 21.1 Iris dataset
Let's explore the iris dataset. The dataset contains measurements of leaves and sepals which can be used to classify the iris flowers in 3 species: Setosa, Versicolour, and Virginica. Feel free to explore the dataset as you see fit. If you'd prefer some guidance, you can instead follow some/all of the proposed steps below.

Below we load the dataset and add a *target_name* column to the `frame` attribute of the dataset. This column can come in handy for certain representations of the data.

In [ ]:
from sklearn.datasets import load_iris
iris_dataset = load_iris(as_frame=True)

target_name_dict = dict(enumerate(iris_dataset.target_names))
iris_dataset.frame["target name"] = iris_dataset.frame["target"].map(lambda x: target_name_dict[x])

### 21.1.1 First look at the dataset
Take a look at the `iris_dataset` object. You can look at `iris_dataset.DESCR` to see the description (print it for readability). `iris_dataset.keys()` will show you available objects in the dataset, note that `frame` contains both the features (`iris.data`) and the target (`iris.target`).

In [ ]:
iris_dataset.data.columns

In [ ]:
print(iris_dataset.DESCR)

### 21.1.2 Feature distributions
Let's take a look at the feature distributions. Plot the distribution of each feature, with separate histograms for the different iris species.

In [ ]:
# With matplotlib
for feature in iris_dataset.feature_names:
    plt.figure()
    plt.hist([iris_dataset.frame[feature][iris_dataset.frame["target name"]==name] for name in iris_dataset.target_names], label=feature, bins=15, density=True)
    plt.title(feature)
    plt.xlabel("Size [cm]")
    plt.ylabel("Density")
    plt.show()

In [ ]:
# With matplotlib and pandas'groupby method.
grouped = iris_dataset.frame.groupby("target name")
for feature in iris_dataset.feature_names:
    plt.figure()
    plt.hist([group[feature] for group_id, group in grouped], label = [group_id for group_id, group in grouped])
    plt.legend(loc="best")
    plt.show()

In [ ]:
# With seaborn
for feature in iris_dataset.feature_names:
    sns.histplot(iris_dataset.frame, x=feature, kde=True, bins=15, stat="density", hue="target name")
    plt.show()

In [ ]:
# Using pandas'pivot and hist methods
for feature in iris_dataset.feature_names:
    iris_dataset.frame.pivot(columns="target name")[feature].plot.hist()

### 21.1.2 Correlation matrix
Let's now take a look at the correlation matrix (for `iris_dataset.frame` but without the last column containing the *target_name*), to see the correlation between the feature but also with the target. You can plot the correlation matrix using seaborn's `heatmap` function, `sns.heatmap`.

In [ ]:
correlation_mat = iris_dataset.frame.iloc[:,:-1].corr()
sns.heatmap(correlation_mat, vmin=-1, vmax=1, cmap="bwr")
plt.grid(False)

### 21.1.3 one-way ANOVA
Correlation is not appropriate for categorical variables. Instead let's do an ANOVA to test whether the different species have the same means for the different features or not. For this we can use `f_oneway` from `scipy.stats`. You'll need to perform one ANOVA per feature.

In [ ]:
from scipy.stats import f_oneway
groups = iris_dataset.frame.groupby("target")
for feature in iris_dataset.feature_names:
    result = f_oneway([group[feature] for groupid, group in groups], axis=1)
    print(feature, result)

In [ ]:
# Same as above but without a for loop, as f_oneway also allows testing for multiple features.
groups = iris_dataset.frame.groupby("target")
result = f_oneway([group.iloc[:,:4] for groupid, group in groups], axis=1)
print(result)

We see that means are significantly different between species for all features.

### 21.1.4 Principal Components Analysis
We could now make a Principal Components Analysis and plot the different species on the first two vectors. For this you can use the `PCA` from the `sklearn.decomposition` module.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA()
pca_transformed = pca.fit_transform(iris_dataset.data)
pca_transformed = pd.DataFrame(pca_transformed)
pca_transformed["target name"] = iris_dataset.frame["target name"]

In [ ]:
groups = pca_transformed.groupby("target name")
plt.figure()
for groupid, group in pca_transformed.groupby("target name"):
    plt.scatter(group[0], group[1], label=groupid)
plt.xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)")
plt.ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)")
plt.legend(loc="best")
plt.show()


Seeing how well the clusters are separated it should be fairly easy to make a good predictor of the species based on the features.

### 21.1.5 Predictor

#### 21.1.5.1 Build a predictor
Let's make a simple predictor of species based on our 4 features.

We will start simple, splitting the data into a training and test set, and then fitting an `SVC` after normalizing the features with the `StandardScaler`. We can use a `Pipeline` for easier integration of the two steps in a single object. This also allows to avoid data leakage when doing cross-validation or hyper-parameter optimization.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(iris_dataset.data, iris_dataset.target, stratify=iris_dataset.target)

In [ ]:
svc_pipe = Pipeline([('scaler', StandardScaler()), ('svc', SVC(probability=True))])
svc_pipe.fit(X_train, y_train)
svc_pipe.score(X_test, y_test)

To get an idea of how sensitive the model is on data split, we could instead do cross-validation

In [ ]:
scores = cross_validate(svc_pipe, iris_dataset.data, iris_dataset.target)
pp(scores)

We could also try to optimize the hyper-parameters of the SVC, notably C and the kernel. For this we will use cross valiation in combination with a grid search: `GridSearchCV`

In [ ]:
from sklearn.model_selection import GridSearchCV
hyper_param_search_space = {'svc__C': [0.001, 0.1, 1, 10, 100], 'svc__kernel': ['linear', 'poly', 'rbf', 'sigmoid']}
clf = GridSearchCV(svc_pipe, param_grid=hyper_param_search_space)
search = clf.fit(X_train, y_train)

In [ ]:
print("Best score on training data", search.best_score_)
print("Best hyper-parameters", search.best_params_)
print("Score on test data", search.best_estimator_.score(X_test, y_test))

#### 21.1.5.2 Assess predictor quality
We will now plot the confusion matrix using `sklearn.metrics.ConfusionMatrixDisplay`, show a classification report using `sklearn.metrics.classification_report` and show the ROC curves using `from sklearn.metrics.RocCurveDisplay`, but this last step is fairly complex for multiclass predictions.

We will use the simple SVC classifier we've made, as prediction quality was the same as for the optimized predictors.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
predicted_target = svc_pipe.predict(X_test)
target_names = iris_dataset.target_names
ConfusionMatrixDisplay.from_predictions(
    y_test, predicted_target, normalize="true", cmap="Blues", 
    display_labels=target_names)
plt.grid(False)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, predicted_target, target_names=target_names))

For the ROC curves we need the predicted probabilites for the target, for this we need to have set `probability=True` in the initialization of our `SVC` classifier above (which we did).

In [ ]:
from sklearn.metrics import RocCurveDisplay
from sklearn.preprocessing import LabelBinarizer
predicted_proba = svc_pipe.predict_proba(X_test)
target_one_hot_encoded = LabelBinarizer().fit_transform(y_test)

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(
    target_one_hot_encoded.ravel(),
    predicted_proba.ravel(),
    name="micro-average OvR",
    curve_kwargs=dict(color="r"),
    plot_chance_level=True,
    despine=True,
    ax=ax
)

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for class_id, color in zip(range(target_one_hot_encoded.shape[1]), colors):
    RocCurveDisplay.from_predictions(
        target_one_hot_encoded[:, class_id],
        predicted_proba[:, class_id],
        drop_intermediate=False,
        name=f"{target_names[class_id]}",
        curve_kwargs=dict(color=color, linestyle="--"),
        ax=ax,
        despine=True,
    )
_ = ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title=" One-vs-Rest ROC",
)

### 21.1.6 Feature engineering
Intuitively it could make sense to add petal and sepal area as features. Let's do this and then redo a PCA to visualize the clusters and train a new classifier for comparison.

In [ ]:
iris_dataset.data["petal surface (cm)"] = iris_dataset.data["petal length (cm)"] * iris_dataset.data["petal width (cm)"]
iris_dataset.data["sepal surface (cm)"] = iris_dataset.data["sepal length (cm)"] * iris_dataset.data["sepal width (cm)"]


In [ ]:
from sklearn.decomposition import PCA
pca = PCA()
pca_transformed = pca.fit_transform(iris_dataset.data)
pca_transformed = pd.DataFrame(pca_transformed)
pca_transformed["target name"] = iris_dataset.frame["target name"]

groups = pca_transformed.groupby("target name")
plt.figure()
for groupid, group in pca_transformed.groupby("target name"):
    plt.scatter(group[0], group[1], label=groupid)
plt.xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)")
plt.ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)")
plt.legend(loc="best")
plt.show()

And let's generate a new estimator

In [ ]:
scores = cross_validate(svc_pipe, iris_dataset.data, iris_dataset.target)
pp(scores)

Prediction quality did not improve.

## 21.2 Forest cover dataset

We will now look at a dataset of forest cover types. As before feel free to explore the dataset on your own or to follow the steps outlined below. Let us load the dataset and add labels for the target values (there are 7 cover types):

In [ ]:
from sklearn.datasets import fetch_covtype
covertype_dataset = fetch_covtype(as_frame=True)

target_names = [
    "Spruce/Fir", 
    "Lodgepole Pine", 
    "Ponderosa Pine", 
    "Cottonwood/Willow", 
    "Aspen", 
    "Douglas-fir", 
    "Krummholz"
]
target_name_dict = dict(enumerate(target_names, 1))
covertype_dataset.target_names = np.array(list(map(lambda x: target_name_dict[x], covertype_dataset.target)))
covertype_dataset.frame["Cover_Type_Name"] = covertype_dataset.target_names

### 21.2.1 First look at the dataset

Let's have a first look at the dataset, i.e. look at its description (`DESCR` attribute) and have a look at the features in the dataset. Note that the `Wilderness_Area_i` features as well as the `Soil_Type_i` features are hot encodings, i.e. each sample is in one wilderness area and has one soil type.

In [ ]:
print(covertype_dataset.DESCR)

In [ ]:
covertype_dataset.data.describe()

### 21.2.2 Comparison of wilderness areas

We start by looking at the different wilderness areas and check how some of the features and cover type are distributed in the different wilderness areas:
- group the data by area
- calculate the means using `mean(numeric_only=True)` on the groups to avoid an error due to taking the mean of non-numerical features). 
- For the `Cover_Type_Name`, you could instead use `value_counts(normalize=True)` on the groups to get the fraction of each cover type seen in each wilderness area

For this we first need to inverse the one hot encoding of the wilderness areas. As this is not straightforward I did it for you below:

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(categories=[["Rawah", "Neota", "Comanche Peak", "Cache la Poudre"]]).fit(np.array(["Rawah", "Neota", "Comanche Peak", "Cache la Poudre"]).reshape(-1, 1))
covertype_dataset.frame["wilderness_area"] = [el[0] for el in encoder.inverse_transform(covertype_dataset.data.iloc[:,10:14])]

In [ ]:
groups = covertype_dataset.frame.groupby("wilderness_area")
groups.mean(numeric_only=True).iloc[:,:10]

In [ ]:
groups["Cover_Type_Name"].value_counts(normalize=True)

### 21.2.3 Classifier

#### 21.2.3.1 Build a classifier
Let's now build a classifier: A `sklearn.ensemble.RandomForestClassifier` yields good results with a decent fitting time (you can set the number of parallel processes used for fitting by passing `n_jobs=10` when initializing the classifier). If you want something faster to fit you can use a `sklearn.tree.DecisionTreeClassifier` instead, but this will yield ROC curves that only have 3 points and therefore look a bit weird.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
classifier = RandomForestClassifier(class_weight="balanced", n_jobs=12, n_estimators=100)
predicted_target = cross_val_predict(classifier, covertype_dataset.data, covertype_dataset.target)

#### 21.2.3.2 Assess classification quality
As for the iris dataset we will show the classification report, display the confusion matrix and plot the ROC curves.

In [ ]:
print(classification_report(covertype_dataset.target, predicted_target, target_names=target_names))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    covertype_dataset.target, predicted_target, normalize="true", 
    cmap="Blues", display_labels=target_names)
plt.grid(False)

In [ ]:
predicted_proba = cross_val_predict(classifier, covertype_dataset.data, covertype_dataset.target, method="predict_proba")
target_one_hot_encoded = LabelBinarizer().fit_transform(covertype_dataset.target)

fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(
    target_one_hot_encoded.ravel(),
    predicted_proba.ravel(),
    name="micro-average OvR",
    curve_kwargs=dict(color="r"),
    plot_chance_level=True,
    despine=True,
    ax=ax
)

colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for class_id, color in zip(range(target_one_hot_encoded.shape[1]), colors):
    RocCurveDisplay.from_predictions(
        target_one_hot_encoded[:, class_id],
        predicted_proba[:, class_id],
        drop_intermediate=False,
        name=f"{target_names[class_id]}",
        curve_kwargs=dict(color=color, linestyle="--"),
        ax=ax,
        despine=True,
    )
_ = ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title=" One-vs-Rest ROC",
)

We could have simply fit a `DecisionTreeClassifier` instead, which leads to somewhat lower classification scores, but is less complex and faster to fit.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_predict
classifier = DecisionTreeClassifier(max_depth=40)
predicted_target = cross_val_predict(classifier, covertype_dataset.data, covertype_dataset.target)
print(classification_report(covertype_dataset.target, predicted_target, target_names=target_names))